In [78]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

In [79]:
receivals = pd.read_csv('data/kernel/receivals.csv', parse_dates = ['date_arrival']).copy()
purchase_orders = pd.read_csv("data/kernel/purchase_orders.csv", parse_dates=["delivery_date", "created_date_time", "modified_date_time"]).copy()
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)
materials = pd.read_csv("data/extended/materials.csv").copy()

In [80]:
purchase_orders = purchase_orders[purchase_orders['created_date_time'].dt.year > 2018]
receivals = receivals[receivals['date_arrival'].dt.year > 2018]
""" receivals = receivals[~((receivals["date_arrival"].dt.month == 2) & (receivals["date_arrival"].dt.day == 29) & (receivals["date_arrival"].dt.year == 2024))]
purchase_orders = purchase_orders[~((purchase_orders["created_date_time"].dt.month == 2) & (purchase_orders["created_date_time"].dt.day == 29) & (purchase_orders["created_date_time"].dt.year == 2024))] """

purchase_orders_clean = purchase_orders[purchase_orders['status'] != 'Deleted']
receivals_clean = receivals[receivals['net_weight'] > 0]

data = receivals_clean.merge(purchase_orders_clean, how='inner', on=['purchase_order_id', 'purchase_order_item_no', 'product_id'], suffixes=('_receival', '_order'))

print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34435 entries, 0 to 34434
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   rm_id                   34435 non-null  float64            
 1   product_id              34435 non-null  float64            
 2   purchase_order_id       34435 non-null  float64            
 3   purchase_order_item_no  34435 non-null  float64            
 4   receival_item_no        34435 non-null  int64              
 5   batch_id                34435 non-null  float64            
 6   date_arrival            34435 non-null  datetime64[ns]     
 7   receival_status         34435 non-null  object             
 8   net_weight              34435 non-null  float64            
 9   supplier_id             34435 non-null  int64              
 10  quantity                34435 non-null  float64            
 11  delivery_date           34435 non-null  o

In [81]:
# Fjerner stock_location 'DELETED' i materials
materials['product_id'] = pd.to_numeric(materials['product_id'], errors='coerce')
materials['rm_id'] = pd.to_numeric(materials['rm_id'], errors='coerce')

active_materials = materials[~materials['stock_location'].astype(str).str.upper().str.contains('DELETED')].copy()

# 2) Tillatte par etter filtrering
allowed_pairs = (
    active_materials[['product_id', 'rm_id']]
    .dropna()
    .drop_duplicates()
)

# 3) Harmoniser typer i data og filtrer med membership i stedet for merge for å unngå duplikater
data['product_id'] = pd.to_numeric(data['product_id'], errors='coerce')
data['rm_id'] = pd.to_numeric(data['rm_id'], errors='coerce')

allowed_index = pd.MultiIndex.from_frame(allowed_pairs)
current_index = pd.MultiIndex.from_frame(data[['product_id', 'rm_id']])
mask = current_index.isin(allowed_index)

n_before = len(data)
data = data.loc[mask].copy()
removed_rows = n_before - len(data)

print(f"Kept only non-DELETED pairs: removed {removed_rows} rows (from {n_before} to {len(data)}).")

Kept only non-DELETED pairs: removed 2 rows (from 34435 to 34433).


In [82]:
df = data.copy()
for c in ["date_arrival", "created_date_time", "delivery_date"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None)
# Beregner faktisk differanse i dager i stedet for å slå opp .dt.days direkte
df['lead_time_days'] = (df['date_arrival'] - df['created_date_time']).dt.days
df = df[df['lead_time_days'] >= 0]



In [83]:
mapping = pd.read_csv("data/prediction_mapping.csv", parse_dates=["forecast_start_date", "forecast_end_date"]).copy()

In [84]:
daily = (
    df.assign(date=df['date_arrival'].dt.normalize())
      .groupby(['rm_id', 'date'], as_index=False)
      .agg(total_weight=('net_weight', 'sum'),
           lead_time_days=('lead_time_days', 'mean'))
)

daily['doy'] = daily['date'].dt.dayofyear
daily['month'] = daily['date'].dt.month
daily['dow'] = daily['date'].dt.weekday
daily['is_weekend'] = (daily['dow'] >= 5).astype(int)
daily['year'] = daily['date'].dt.year

train_daily = daily.loc[daily['year'] <= 2024].copy()
train_daily = train_daily.dropna(subset=['rm_id']).copy()

rm_totals_2024 = (
    train_daily.loc[train_daily['year'] == 2024]
    .groupby('rm_id', as_index=False)['total_weight']
    .sum()
    .rename(columns={'total_weight': 'total_weight_2024'})
)
rm_ids_2024 = rm_totals_2024['rm_id'].tolist()
print(f"rm_id med leveranser i 2024: {len(rm_ids_2024)}")
if not rm_ids_2024:
    raise RuntimeError("Fant ingen rm_id med leveranser i 2024. Kan ikke trene modell.")

TOP_N = 45
if TOP_N is not None and len(rm_ids_2024) > TOP_N:
    top_rm_ids = rm_totals_2024.nlargest(TOP_N, 'total_weight_2024')['rm_id'].tolist()
    removed_count = len(rm_ids_2024) - len(top_rm_ids)
    print(f"Beholder top {len(top_rm_ids)} rm_id med leveranser i 2024 (TOP_N={TOP_N}). Filtrert bort {removed_count} øvrige rm.")
else:
    top_rm_ids = rm_ids_2024
    print(f"Bruker alle {len(top_rm_ids)} rm_id med leveranser i 2024 (<=TOP_N).")

train_daily = train_daily.loc[train_daily['rm_id'].isin(top_rm_ids)].copy()

# Fyll manglende lead-time per rm_id, og fall tilbake til global median
train_daily['lead_time_days'] = train_daily.groupby('rm_id')['lead_time_days'].transform(
    lambda s: s.fillna(s.median())
)
global_lead_time = float(train_daily['lead_time_days'].median()) if not train_daily['lead_time_days'].dropna().empty else 0.0
train_daily['lead_time_days'] = train_daily['lead_time_days'].fillna(global_lead_time)

rm_ids_train = np.sort(train_daily['rm_id'].dropna().unique())
rm_encoder = {rm: idx for idx, rm in enumerate(rm_ids_train)}
train_daily['rm_code'] = train_daily['rm_id'].map(rm_encoder).astype('int32')

def compute_temporal_features(frame: pd.DataFrame, value_col: str) -> pd.DataFrame:
    frame = frame.sort_values(['rm_id', 'date']).copy()
    frame['__value_shift'] = frame.groupby('rm_id')[value_col].shift(1).fillna(0.0)
    features = frame[['rm_id', 'date']].copy()
    
    for window in (90, 180):
        col_name = f"sum_past_{window}"
        parts = []
        for rm, sub in frame.groupby('rm_id'):
            sub_sorted = sub[['date', '__value_shift']].copy()
            rolling_sum = (
                sub_sorted
                .set_index('date')['__value_shift']
                .rolling(f'{window}D', min_periods=1)
                .sum()
            )
            parts.append(pd.DataFrame({
                'rm_id': rm,
                'date': sub['date'].values,
                col_name: rolling_sum.to_numpy()
            }))
        window_df = pd.concat(parts, ignore_index=True)
        features = features.merge(window_df, on=['rm_id', 'date'], how='left')

    features['ratio_90_180'] = features['sum_past_90'] / (features['sum_past_180'] + 1e-6)
    features['diff_90_180'] = features['sum_past_90'] - features['sum_past_180']

    lag_lookup = frame[['rm_id', 'date', value_col]].copy()
    lag_lookup['date'] = lag_lookup['date'] + pd.Timedelta(days=365)
    lag_lookup = lag_lookup.rename(columns={value_col: 'lag_365'})
    features = features.merge(lag_lookup, on=['rm_id', 'date'], how='left')

    for col in ['sum_past_90', 'sum_past_180', 'ratio_90_180', 'diff_90_180', 'lag_365']:
        features[col] = features[col].fillna(0.0)

    return features

temporal_features_train = compute_temporal_features(
    train_daily[['rm_id', 'date', 'total_weight']].copy(),
    value_col='total_weight'
 )
train_daily = train_daily.merge(temporal_features_train, on=['rm_id', 'date'], how='left')

train_daily['doy_sin'] = np.sin(2 * np.pi * train_daily['doy'] / 365.0)
train_daily['doy_cos'] = np.cos(2 * np.pi * train_daily['doy'] / 365.0)
train_daily['dow_sin'] = np.sin(2 * np.pi * train_daily['dow'] / 7.0)
train_daily['dow_cos'] = np.cos(2 * np.pi * train_daily['dow'] / 7.0)

train_daily = train_daily.replace([np.inf, -np.inf], 0.0)

rm_lead_time_lookup = train_daily.groupby('rm_id')['lead_time_days'].median()

rm_id med leveranser i 2024: 51
Beholder top 45 rm_id med leveranser i 2024 (TOP_N=45). Filtrert bort 6 øvrige rm.


In [85]:
feature_cols = ['rm_code', 'doy', 'doy_sin', 'doy_cos', 'month', 'dow', 'dow_sin', 'dow_cos', 'is_weekend', 'lead_time_days', 'sum_past_90', 'sum_past_180', 'ratio_90_180', 'diff_90_180', 'lag_365']
X_train = train_daily[feature_cols]
y_train = (train_daily['total_weight'])

y_train_log = np.log1p(y_train)

In [86]:
model = LGBMRegressor(
    objective='quantile',
    alpha = 0.1,
    n_estimators=10000,
    learning_rate=0.05,
    max_depth= 6,
    random_state=42,
    verbose = -1
)
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
train_rmse = float(np.sqrt(np.mean((y_train - train_pred) ** 2)))
print(f"Train RMSE (daglig totalvekt): {train_rmse:.2f}")

first_start = mapping['forecast_start_date'].min()
last_end = mapping['forecast_end_date'].max()
if pd.isna(first_start) or pd.isna(last_end):
    raise RuntimeError("Mapping mangler gyldige start/slutt datoer.")

calendar = pd.date_range(first_start - pd.Timedelta(days=1), last_end, freq='D')
rm_ids_future = mapping.loc[mapping['rm_id'].isin(top_rm_ids), 'rm_id'].dropna().unique()
print(f"rm_id i 2025-grid etter filtrering: {rm_ids_future.size}")
grid = (
    pd.MultiIndex.from_product([rm_ids_future, calendar], names=['rm_id', 'date'])
    .to_frame(index=False)
)

grid['rm_code'] = grid['rm_id'].map(rm_encoder)
grid['lead_time_days'] = grid['rm_id'].map(rm_lead_time_lookup)
grid['lead_time_days'] = grid['lead_time_days'].fillna(global_lead_time)

history_weights = daily.loc[daily['rm_id'].isin(top_rm_ids), ['rm_id', 'date', 'total_weight']].copy()
grid = grid.merge(history_weights, on=['rm_id', 'date'], how='left')
grid['historical_weight'] = grid['total_weight'].fillna(0.0)
grid = grid.drop(columns=['total_weight'])

temporal_features_grid = compute_temporal_features(
    grid[['rm_id', 'date', 'historical_weight']].copy(),
    value_col='historical_weight'
 )
grid = grid.merge(temporal_features_grid, on=['rm_id', 'date'], how='left')

grid['doy'] = grid['date'].dt.dayofyear
grid['month'] = grid['date'].dt.month
grid['dow'] = grid['date'].dt.weekday
grid['is_weekend'] = (grid['dow'] >= 5).astype(int)
grid['doy_sin'] = np.sin(2 * np.pi * grid['doy'] / 365.0)
grid['doy_cos'] = np.cos(2 * np.pi * grid['doy'] / 365.0)
grid['dow_sin'] = np.sin(2 * np.pi * grid['dow'] / 7.0)
grid['dow_cos'] = np.cos(2 * np.pi * grid['dow'] / 7.0)

grid[['sum_past_90', 'sum_past_180', 'ratio_90_180', 'diff_90_180', 'lag_365']] = (
    grid[['sum_past_90', 'sum_past_180', 'ratio_90_180', 'diff_90_180', 'lag_365']].fillna(0.0)
)

grid = grid.replace([np.inf, -np.inf], 0.0)

known_mask = grid['rm_code'].notna()
unknown_rm = grid.loc[~known_mask, 'rm_id'].unique()
if len(unknown_rm) > 0:
    print(f"Advarsel: {len(unknown_rm)} rm_id mangler historikk. Setter prediksjon til 0 for disse.")

grid['predicted_daily_weight'] = 0.0
if known_mask.any():
    known_features = grid.loc[known_mask, feature_cols].copy()
    known_features['rm_code'] = known_features['rm_code'].astype('int32')
    known_features = known_features.fillna(0.0)
    grid.loc[known_mask, 'predicted_daily_weight'] = model.predict(known_features)

grid.loc[grid['date'] == first_start - pd.Timedelta(days=1), 'predicted_daily_weight'] = 0.0
grid['predicted_daily_weight'] = grid['predicted_daily_weight'].clip(lower=0)

grid = grid.sort_values(['rm_id', 'date']).reset_index(drop=True)
grid['cumulative_weight'] = grid.groupby('rm_id')['predicted_daily_weight'].cumsum()

mapping_table = mapping[['ID', 'rm_id', 'forecast_start_date', 'forecast_end_date']].copy()
mapping_table = mapping_table.loc[mapping_table['rm_id'].isin(rm_ids_future)].copy()
mapping_table = mapping_table.merge(
    grid.rename(columns={'date': 'forecast_end_date', 'cumulative_weight': 'cum_end'})[
        ['rm_id', 'forecast_end_date', 'cum_end']
    ],
    on=['rm_id', 'forecast_end_date'],
    how='left'
)
mapping_table['start_prev'] = mapping_table['forecast_start_date'] - pd.Timedelta(days=1)
mapping_table = mapping_table.merge(
    grid.rename(columns={'date': 'start_prev', 'cumulative_weight': 'cum_prev'})[
        ['rm_id', 'start_prev', 'cum_prev']
    ],
    on=['rm_id', 'start_prev'],
    how='left'
)
mapping_table['cum_prev'] = mapping_table['cum_prev'].fillna(0.0)
mapping_table['predicted_weight'] = (mapping_table['cum_end'].fillna(0.0) - mapping_table['cum_prev']).clip(lower=0)

submission_2025 = (
    mapping[['ID', 'rm_id']]
    .merge(
        mapping_table[['ID', 'predicted_weight']],
        on='ID',
        how='left'
)
)
submission_2025['predicted_weight'] = submission_2025['predicted_weight'].fillna(0.0)
submission_2025 = submission_2025.sort_values('ID').reset_index(drop=True)

# --- Diagnose: nøkkeltall for prediksjonene ---
total_rm = mapping['rm_id'].nunique()
rm_pred = submission_2025.groupby('rm_id', as_index=False)['predicted_weight'].sum()
rm_pred_positive = rm_pred.loc[rm_pred['predicted_weight'] > 0]
num_rm_positive = rm_pred_positive['rm_id'].nunique()
print(f"rm_id med positiv totalvekt i 2025: {num_rm_positive} / {total_rm}")

total_weight_2025 = (submission_2025['predicted_weight'].sum()*0.25)
print(f"Total prognosert vekt (kg) 2025-perioden: {total_weight_2025:,.0f}")

rm_without_history = mapping.loc[~mapping['rm_id'].isin(rm_ids_future), 'rm_id'].nunique()
print(f"rm_id uten historikk eller utenfor TOP_N (setter 0): {rm_without_history}")
if len(unknown_rm) > 0:
    print(f"rm_id innenfor TOP_N men uten historikk (setter 0): {len(unknown_rm)}")

submission_2025[['ID', 'predicted_weight']].to_csv('submission_2025_direct.csv', index=False)
print(f"Lagring fullført: {len(submission_2025)} rader i submission_2025_direct.csv")

Train RMSE (daglig totalvekt): 21254.86
rm_id i 2025-grid etter filtrering: 45
rm_id med positiv totalvekt i 2025: 45 / 203
Total prognosert vekt (kg) 2025-perioden: 1,946,277,682
rm_id uten historikk eller utenfor TOP_N (setter 0): 158
Lagring fullført: 30450 rader i submission_2025_direct.csv
rm_id med positiv totalvekt i 2025: 45 / 203
Total prognosert vekt (kg) 2025-perioden: 1,946,277,682
rm_id uten historikk eller utenfor TOP_N (setter 0): 158
Lagring fullført: 30450 rader i submission_2025_direct.csv
